# FemurFix v2 — YOLO11s at 960 px

Second training iteration after diagnosing the v1 model's low sensitivity (image-level recall 54%
on the held-out FracAtlas test split at conf 0.15; hip/proximal cases near zero).

Changes vs v1:
- YOLO11s instead of YOLO11n (more capacity for subtle cortical lines)
- 960 px training instead of 640 (fracture boxes are small; median 0.4% of image area)
- horizontal flip and small rotations enabled (v1 had flips disabled)
- 80 epochs with 20-epoch patience
- ONNX exported at 960 for backend deployment

This is still FracAtlas (mixed anatomy). It is an incremental sensitivity improvement, not a
substitute for a femur-focused, region-labelled dataset.

Expected runtime: roughly 1.5–2.5 h on a T4. If CUDA OOM, set batch to 4.


In [ ]:
import subprocess, sys, torch
print("Python:", sys.version.split()[0])
print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
else:
    print("WARNING: no GPU; enable a T4 accelerator in notebook settings.")


In [ ]:
%pip install -q "ultralytics>=8.3,<9" "albumentations>=1.4.21,<2" "onnx>=1.16" "onnxslim>=0.1.31"


In [ ]:
from pathlib import Path
import os, shutil

CANDIDATES = [
    Path("/kaggle/input/femurfix-fracatlas-yolo"),
    Path("/kaggle/input/datasets/birwatkar/femurfix-fracatlas-yolo"),
]
DATA_SRC = next((p for p in CANDIDATES if (p / "images" / "train").is_dir()), None)
if DATA_SRC is None:
    for images_dir in Path("/kaggle/input").rglob("images"):
        if images_dir.is_dir() and (images_dir / "train").is_dir() and (images_dir.parent / "labels" / "train").is_dir():
            DATA_SRC = images_dir.parent
            break
if DATA_SRC is None:
    raise FileNotFoundError("Attach the femurfix-fracatlas-yolo dataset (images/ and labels/).")

WORK = Path("/kaggle/temp/fracatlas-v2")
if not (WORK / "images" / "train").is_dir():
    print(f"Copying {DATA_SRC} -> {WORK} ...")
    shutil.copytree(DATA_SRC, WORK, dirs_exist_ok=True)
DATA_ROOT = WORK
print("Dataset root:", DATA_ROOT)
for split in ("train", "val", "test"):
    n_img = len(list((DATA_ROOT / "images" / split).glob("*")))
    n_lbl = len(list((DATA_ROOT / "labels" / split).glob("*.txt")))
    print(f"  {split}: {n_img} images / {n_lbl} labels")


In [ ]:
from pathlib import Path
RUN_ROOT = Path("/kaggle/working/runs")
RUN_ROOT.mkdir(parents=True, exist_ok=True)
DATA_YAML = RUN_ROOT / "fracatlas_data.yaml"
DATA_YAML.write_text(f'''path: {DATA_ROOT.resolve().as_posix()}
train: images/train
val: images/val
test: images/test

names:
  0: fracture
''')
print(DATA_YAML.read_text())


In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11s.pt")
model.train(
    data=str(DATA_YAML),
    epochs=80,
    patience=20,
    imgsz=960,
    batch=8,
    device=0,
    workers=4,
    project=str(RUN_ROOT),
    name="fracatlas_yolo11s_960",
    exist_ok=True,
    seed=0,
    deterministic=True,
    optimizer="auto",
    amp=True,
    cache=True,
    plots=True,
    save=True,
    val=True,
    close_mosaic=12,
    hsv_h=0.0,
    hsv_s=0.0,
    hsv_v=0.4,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    flipud=0.0,
    degrees=10.0,
    mosaic=1.0,
    mixup=0.0,
    copy_paste=0.0,
    erasing=0.0,
    auto_augment=None,
)
print("best:", model.trainer.best)


In [ ]:
import json
from pathlib import Path
from ultralytics import YOLO

run_dir = Path("/kaggle/working/runs/fracatlas_yolo11s_960")
best = Path(run_dir / "weights" / "best.pt")
assert best.is_file(), f"missing {best}"

metrics = YOLO(str(best)).val(
    data=str(DATA_YAML),
    split="test",
    imgsz=960,
    batch=8,
    device=0,
    workers=4,
    project=str(RUN_ROOT),
    name="fracatlas_yolo11s_960_test",
    exist_ok=True,
    plots=True,
)
summary = {
    "test": {
        "precision": float(metrics.box.mp),
        "recall": float(metrics.box.mr),
        "mAP50": float(metrics.box.map50),
        "mAP50-95": float(metrics.box.map),
    },
    "config": {"model": "yolo11s.pt", "imgsz": 960, "epochs": 80, "batch": 8, "fliplr": 0.5, "degrees": 10},
}
Path("/kaggle/working/run_summary_v2.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))


In [ ]:
from pathlib import Path
from ultralytics import YOLO

best = Path("/kaggle/working/runs/fracatlas_yolo11s_960/weights/best.pt")
onnx_path = Path(YOLO(str(best)).export(format="onnx", imgsz=960, opset=12, simplify=True, dynamic=False))
print("exported:", onnx_path)
target = Path("/kaggle/working/femur_fracture_v2.onnx")
target.write_bytes(onnx_path.read_bytes())
Path("/kaggle/working/best_v2.pt").write_bytes(best.read_bytes())
print("download-ready:", target, target.stat().st_size, "bytes")


## Outputs
- `/kaggle/working/femur_fracture_v2.onnx` — deployment candidate
- `/kaggle/working/best_v2.pt` — checkpoint
- `/kaggle/working/run_summary_v2.json` — held-out test metrics
- `/kaggle/working/runs/...` — curves, confusion matrix, predictions

Integrate only if v2 beats v1 on the held-out test split and on the femur sample set.
